# Split feasibility

Compare provisional common subject folds across tasks. Set `COHORTS` to select years.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.eda.io import load_tasks, cohort_tables

COHORTS = None  # Discover all years; use ["1year"] or ["2year", "3year"] to restrict.
tables = load_tasks(ROOT / "data", cohorts=COHORTS)
pd.set_option("display.max_rows", 24)
pd.set_option("display.max_columns", 12)
print("Loaded task files by cohort:", {c: len(g) for c, g in cohort_tables(tables).items()})


Loaded task files by cohort: {'1year': 21, '2year': 21, '3year': 18}


Allocate cohorts separately, with one assignment shared across tasks. Include all subjects; only labels `{0, 1, 2}` contribute class indicators. Missing task records contribute none. `-1` remains unresolved.

In [2]:
from src.core.schema import EDA_FOLD_COUNTS, EDA_CANDIDATE_SEEDS, EDA_MIN_FOLD_SUPPORT
from src.eda.splits import theoretical_feasibility, evaluate_candidates, candidate_comparison
from src.eda.audit import audit_tasks

audit = audit_tasks(tables)
print("Candidate fold counts:", EDA_FOLD_COUNTS, "Fixed seeds:", EDA_CANDIDATE_SEEDS)
print("Descriptive minimum-support warning:", EDA_MIN_FOLD_SUPPORT, "subjects per class per partition")
display(audit.cohorts)
feasibility_parts, feasibility_warnings = [], []
for cohort, group in cohort_tables(tables).items():
    try:
        feasibility_parts.append(theoretical_feasibility(group))
    except (ValueError, KeyError) as error:
        feasibility_warnings.append({"cohort": cohort, "reason": str(error)})
feasibility = pd.concat(feasibility_parts, ignore_index=True) if feasibility_parts else pd.DataFrame()
if feasibility_warnings:
    display(pd.DataFrame(feasibility_warnings))


Candidate fold counts: (2, 3, 4, 5) Fixed seeds: (42, 43)
Descriptive minimum-support warning: 5 subjects per class per partition


,cohort,task_files,expected_tasks,rows,subjects,subjects_in_every_task,identical_subject_sets,Gender_conflicting_subjects,Age_conflicting_subjects,Dominant_Hand_conflicting_subjects
0,1year,21,21,2935,140,136,False,0,0,0
1,2year,21,21,3045,145,145,True,0,0,0
2,3year,18,18,2610,145,145,True,0,0,0


In [3]:
if not feasibility.empty:
    task_support = feasibility.groupby(["cohort", "folds", "task"], sort=False).agg(
        class_coverage_necessary_condition=("every_test_fold_coverage_possible", "all"),
        any_class_absent=("class_absent", "any"),
        smallest_class=("subject_count", "min"),
        tiny_test_support=("tiny_test_support", "any"),
        tiny_training_support=("tiny_training_support", "any"),
    ).reset_index()
    display(task_support.groupby(["cohort", "folds"], sort=False).agg(
        tasks=("task", "size"),
        tasks_passing_necessary_condition=("class_coverage_necessary_condition", "sum"),
        tasks_with_absent_class=("any_class_absent", "sum"),
        worst_class_support=("smallest_class", "min"),
        tasks_with_tiny_test_support=("tiny_test_support", "sum"),
        tasks_with_tiny_training_support=("tiny_training_support", "sum"),
    ))
    display(feasibility.loc[feasibility.folds.eq(3) & ~feasibility.every_test_fold_coverage_possible,
                            ["cohort", "task", "label", "subject_count", "unavoidable_missing_test_folds"]])


tasks  tasks_passing_necessary_condition  \
cohort folds                                             
1year  2         21                                 16   
       3         21                                 15   
       4         21                                 13   
       5         21                                 13   
2year  2         21                                 16   
       3         21                                 15   
       4         21                                 14   
       5         21                                 13   
3year  2         18                                 13   
       3         18                                 13   
       4         18                                 12   
       5         18                                 11   

              tasks_with_absent_class  worst_class_support  \
cohort folds                                                 
1year  2                            3                    0   
       3                            3                    0   
       4                            3                    0   
       5                            3                    0   
2year  2                            3                    0   
       3                            3                    0   
       4                            3                    0   
       5                            3                    0   
3year  2                            3                    0   
       3                            3                    0   
       4                            3                    0   
       5                            3                    0   

              tasks_with_tiny_test_support  tasks_with_tiny_training_support  
cohort folds                                                                  
1year  2                                10                                10  
       3                                14                                 9  
       4                                16                                 9  
       5                                17                                 9  
2year  2                                10                                10  
       3                                12                                 9  
       4                                15                                 9  
       5                                17                                 9  
3year  2                                 7                                 7  
       3                                 9                                 7  
       4                                12                                 7  
       5                                14                                 7

,cohort,task,label,subject_count,unavoidable_missing_test_folds
3,1year,Task_1,0,2,1
27,1year,Task_3,0,0,3
39,1year,Task_4,0,0,3
195,1year,Task_17,0,0,3
207,1year,Task_18,0,1,2
219,1year,Task_19,0,1,2
255,2year,Task_1,0,2,1
279,2year,Task_3,0,0,3
291,2year,Task_4,0,0,3
447,2year,Task_17,0,0,3


Each class needs at least K subjects for coverage in every test fold; joint coverage can still fail. The seeded greedy allocator balances task-class indicators and keeps fold sizes within one subject. Coverage alone does not establish reliable estimates.

In [4]:
diagnostics, allocation_warnings = evaluate_candidates(tables)
comparison = candidate_comparison(diagnostics)
display(comparison)
if not allocation_warnings.empty:
    display(allocation_warnings)
if not diagnostics.empty:
    assert diagnostics.subject_disjoint.all()
    assert (comparison.test_subjects_max - comparison.test_subjects_min).le(1).all()
    print("Verified one common assignment per candidate, subject disjointness, and balanced subject counts.")


,cohort,folds,seed,subject_disjoint,missing_test_task_class_folds,missing_train_task_class_folds,worst_test_class_support,worst_train_class_support,test_subjects_min,test_subjects_max,absolute_test_difference,absolute_train_difference
0,1year,2,42,True,8,8,0,0,70,70,0.068061,0.068061
1,1year,2,43,True,8,8,0,0,70,70,0.057996,0.057996
2,1year,3,42,True,14,11,0,0,46,47,0.088938,0.048733
3,1year,3,43,True,14,11,0,0,46,47,0.188402,0.095457
4,1year,4,42,True,22,14,0,0,35,35,0.167814,0.052686
5,1year,4,43,True,22,14,0,0,35,35,0.210493,0.069339
6,1year,5,42,True,31,17,0,0,28,28,0.190476,0.051948
7,1year,5,43,True,31,17,0,0,28,28,0.166925,0.043138
8,2year,2,42,True,8,8,0,0,72,73,0.100693,0.100693
9,2year,2,43,True,8,8,0,0,72,73,0.080330,0.080330


Verified one common assignment per candidate, subject disjointness, and balanced subject counts.


In [5]:
if not diagnostics.empty:
    failing = diagnostics.loc[diagnostics.test_class_missing | diagnostics.train_class_missing]
    display(failing.groupby(["cohort", "folds", "task", "label"], sort=False).agg(
        missing_test_folds_across_seeds=("test_class_missing", "sum"),
        missing_train_folds_across_seeds=("train_class_missing", "sum"),
    ).head(20))
    display(diagnostics.groupby(["cohort", "folds"], sort=False).agg(
        smallest_eligible_test=("eligible_test", "min"),
        smallest_eligible_train=("eligible_train", "min"),
        worst_test_class_support=("test_count", "min"),
        worst_training_class_support=("train_count", "min"),
    ))


missing_test_folds_across_seeds  \
cohort folds task    label                                    
1year  2     Task_3  0                                    4   
             Task_4  0                                    4   
             Task_17 0                                    4   
             Task_18 0                                    2   
             Task_19 0                                    2   
       3     Task_1  0                                    2   
             Task_3  0                                    6   
             Task_4  0                                    6   
             Task_17 0                                    6   
             Task_18 0                                    4   
             Task_19 0                                    4   
       4     Task_1  0                                    4   
             Task_3  0                                    8   
             Task_4  0                                    8   
             Task_17 0                                    8   
             Task_18 0                                    6   
             Task_19 0                                    6   
             Task_21 0                                    2   
             Task_16 0                                    2   
       5     Task_3  0                                   10   

                            missing_train_folds_across_seeds  
cohort folds task    label                                    
1year  2     Task_3  0                                     4  
             Task_4  0                                     4  
             Task_17 0                                     4  
             Task_18 0                                     2  
             Task_19 0                                     2  
       3     Task_1  0                                     0  
             Task_3  0                                     6  
             Task_4  0                                     6  
             Task_17 0                                     6  
             Task_18 0                                     2  
             Task_19 0                                     2  
       4     Task_1  0                                     0  
             Task_3  0                                     8  
             Task_4  0                                     8  
             Task_17 0                                     8  
             Task_18 0                                     2  
             Task_19 0                                     2  
             Task_21 0                                     0  
             Task_16 0                                     0  
       5     Task_3  0                                    10

smallest_eligible_test  smallest_eligible_train  \
cohort folds                                                    
1year  2                          52                       52   
       3                          33                       70   
       4                          26                       80   
       5                          20                       86   
2year  2                          57                       57   
       3                          38                       77   
       4                          27                       85   
       5                          22                       93   
3year  2                          56                       56   
       3                          37                       74   
       4                          27                       83   
       5                          22                       92   

              worst_test_class_support  worst_training_class_support  
cohort folds                                                          
1year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0  
2year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0  
3year  2                             0                             0  
       3                             0                             0  
       4                             0                             0  
       5                             0                             0

In [6]:
if not feasibility.empty:
    for cohort in feasibility.cohort.unique():
        conclusions = []
        for k in (3, 5):
            subset = task_support.loc[task_support.cohort.eq(cohort) & task_support.folds.eq(k)]
            impossible = int((~subset.class_coverage_necessary_condition).sum())
            conclusions.append(f"{k}-fold: {impossible} of {len(subset)} tasks fail the necessary class-coverage condition")
        display(Markdown(f"**Observed for {cohort}:** " + "; ".join(conclusions) + "."))


**Observed for 1year:** 3-fold: 6 of 21 tasks fail the necessary class-coverage condition; 5-fold: 8 of 21 tasks fail the necessary class-coverage condition.

**Observed for 2year:** 3-fold: 6 of 21 tasks fail the necessary class-coverage condition; 5-fold: 8 of 21 tasks fail the necessary class-coverage condition.

**Observed for 3year:** 3-fold: 5 of 18 tasks fail the necessary class-coverage condition; 5-fold: 7 of 18 tasks fail the necessary class-coverage condition.

Before fixing folds, confirm IDs and `-1`, resolve incomplete or failed measurements, and set minimum class support. Choose metrics that remain interpretable when classes are absent. Fit preprocessing on training folds. These candidates do not define the final protocol.